# Validate citations against the current search response

A documentation assistant can return a correctly shaped answer with an invented citation.
This example uses `Guard.for_pydantic`, a local custom validator, and per-call metadata
to require every URL in an explicit `citations` field to occur **exactly** in the current
search response. A different path on the same host does not pass.

The default run uses deterministic search and answer fixtures. It needs `guardrails-ai`
in the notebook kernel (`%pip install guardrails-ai` if needed), with no search or generation
credentials. An optional cell at the end performs an anonymous public documentation search
through Unified MCP. Neither path generates an answer with an LLM.

This checks URL membership, not whether a source supports a claim or is still reachable.
For checking text against source content, see the [provenance example](provenance.ipynb).

Community example contributed by George Pickett (Parallel DevRel), prepared with AI
assistance. Related proposal: [#1705](https://github.com/guardrails-ai/guardrails/issues/1705).

In [ ]:
import json

from pydantic import BaseModel, Field

from guardrails import Guard, OnFailAction
from guardrails.errors import ValidationError
from guardrails.validators import FailResult, PassResult, Validator, register_validator


class CitedAnswer(BaseModel):
    answer: str
    # Keep strings unchanged: URL normalization could hide an exact-match failure.
    citations: list[str] = Field(min_length=1)


def source_urls_from(search_response):
    """Read only result URL fields, never URLs found inside excerpts or the answer."""
    return [result["url"] for result in search_response["results"]]


def answer_json(citations):
    # A hand-written candidate answer, standing in for an application's output.
    return json.dumps(
        {
            "answer": "Guard.use can apply a validator to a selected field.",
            "citations": citations,
        }
    )

## A schema alone accepts an invented citation

This small fixture keeps only the result fields used below. In an application, retain the
response used to compose the answer and derive the allowed URLs from that response.
The invented URL also appears in an excerpt: mentioning a link in source text does not
authorize it as a search result citation.

In [ ]:
search_response = {
    "results": [
        {
            "title": "Guardrails README",
            "excerpts": ["Unrelated link: https://example.org/not-returned-by-search"],
            "url": "https://github.com/guardrails-ai/guardrails/blob/main/README.md",
        }
    ]
}
current_urls = source_urls_from(search_response)
returned_url = current_urls[0]
invented_url = "https://example.org/not-returned-by-search"
wrong_path = "https://github.com/guardrails-ai/guardrails/blob/main/not-a-source.md"

schema_guard = Guard.for_pydantic(CitedAnswer)
assert schema_guard.parse(answer_json([invented_url]), num_reasks=0).validation_passed
print("The schema accepts a nonempty citations list containing an invented URL.")

## Add a validator to the citations field

The validator holds no source URLs. `source_urls` is required metadata supplied for each
call, so reusing the guard cannot authorize an earlier response's URLs. Missing metadata
raises Guardrails' native `ValueError` before validation. An empty retrieval supplies an
empty list and fails membership validation.

`NOOP` records a failed outcome without changing the candidate answer. Check
`validation_passed` before displaying `validated_output`. Callers can instead choose
`EXCEPTION`, as shown below. `num_reasks=0` keeps these checks local without an LLM retry.

In [ ]:
@register_validator(name="example/search-citation-membership", data_type="list")
class SearchCitationMembership(Validator):
    required_metadata_keys = ["source_urls"]

    def _validate(self, value, metadata):
        source_urls = metadata["source_urls"]
        if not isinstance(source_urls, list) or not all(
            isinstance(url, str) for url in source_urls
        ):
            return FailResult(
                error_message="source_urls must be a list of URL strings."
            )
        unknown_urls = [url for url in value if url not in source_urls]
        if unknown_urls:
            return FailResult(
                error_message=f"Citations absent from the current search response: {unknown_urls}"
            )
        return PassResult()


guard = Guard.for_pydantic(CitedAnswer).use(
    SearchCitationMembership(on_fail=OnFailAction.NOOP), on="$.citations"
)

result = guard.parse(
    answer_json([returned_url]),
    metadata={"source_urls": source_urls_from(search_response)},
    num_reasks=0,
)
assert result.validation_passed
assert result.validated_output["citations"] == [returned_url]
print(result.validated_output)

## Exercise failure cases through the same Guard

A valid URL does not rescue a mixed list. Empty citations fail the schema; empty search
results cannot authorize any citation. Each call below supplies metadata from its own
retrieval. The assertions make the example executable with the repository's notebook runner.

In [ ]:
cases = [
    ("exact returned URL", [returned_url], current_urls, True),
    ("invented URL", [invented_url], current_urls, False),
    ("wrong path on the same host", [wrong_path], current_urls, False),
    ("mixed citations", [returned_url, invented_url], current_urls, False),
    ("empty citations", [], current_urls, False),
    ("malformed metadata", [returned_url], returned_url, False),
    ("empty retrieval", [returned_url], source_urls_from({"results": []}), False),
]
for label, citations, source_urls, expected in cases:
    outcome = guard.parse(
        answer_json(citations), metadata={"source_urls": source_urls}, num_reasks=0
    )
    assert outcome.validation_passed is expected, label
    if expected:
        assert outcome.validated_output["citations"] == citations
    if not expected and citations:
        assert (
            outcome.validated_output["citations"] == citations
        )  # NOOP preserves input
    print(f"{label}: validation_passed={outcome.validation_passed}")

try:
    guard.parse(answer_json([returned_url]), num_reasks=0)
except ValueError as exc:
    assert "Missing required metadata keys: source_urls" in str(exc)
    print(str(exc))
else:
    raise AssertionError("Missing metadata must raise ValueError")

In [ ]:
# A new response replaces the allowed URLs even though the Guard is reused.
next_response = {
    "results": [
        {
            "url": "https://github.com/guardrails-ai/guardrails/blob/main/guardrails/guard.py"
        }
    ]
}
next_urls = source_urls_from(next_response)
assert not guard.parse(
    answer_json([returned_url]), metadata={"source_urls": next_urls}, num_reasks=0
).validation_passed
assert guard.parse(
    answer_json([next_urls[0]]), metadata={"source_urls": next_urls}, num_reasks=0
).validation_passed
print("The reused guard rejects a stale URL and accepts the current response's URL.")

strict_guard = Guard.for_pydantic(CitedAnswer).use(
    SearchCitationMembership(on_fail=OnFailAction.EXCEPTION), on="$.citations"
)
try:
    strict_guard.parse(
        answer_json([invented_url]),
        metadata={"source_urls": current_urls},
        num_reasks=0,
    )
except ValidationError as exc:
    assert invented_url in str(exc)
    print(str(exc))
else:
    raise AssertionError("EXCEPTION must raise ValidationError")

## Optional: use a current anonymous Unified MCP search response

Leave `RUN_LIVE_SEARCH = False` for an offline run. To opt in, install the optional client
in this kernel with `%pip install "mcp>=1.30,<2"`, change the flag to `True`, and run the
following cells. See [Parallel MCP setup](https://docs.parallel.ai/integrations/mcp/quickstart).
Anonymous `web_search` needs no Parallel key and has lower rate limits.

The client sends a product-specific User-Agent on its HTTP requests and no authorization
header. Authentication, rate-limit, timeout, tool, and malformed-response errors propagate;
they are not converted to empty successful searches. Only public documentation is queried.
The final check constructs a candidate citation from a returned URL to exercise the guard;
it demonstrates live retrieval plus local validation, not live answer generation.

In [ ]:
RUN_LIVE_SEARCH = False

In [ ]:
async def search_public_docs():
    # These imports are needed only when the live search is enabled.
    from datetime import timedelta

    import httpx
    from mcp import ClientSession
    from mcp.client.streamable_http import streamable_http_client

    server_url = "https://mcp.parallel.ai/mcp"
    headers = {
        "User-Agent": "guardrails-citation-example/0.1 (Parallel DevRel validation)"
    }
    async with httpx.AsyncClient(
        headers=headers, timeout=httpx.Timeout(30.0, read=60.0)
    ) as client:
        async with streamable_http_client(server_url, http_client=client) as (
            read,
            write,
            _,
        ):
            async with ClientSession(
                read, write, read_timeout_seconds=timedelta(seconds=60)
            ) as session:
                await session.initialize()
                result = await session.call_tool(
                    "web_search",
                    {
                        "objective": (
                            "Find the public Guardrails documentation or source explaining "
                            "Guard.use and custom validator runtime metadata."
                        ),
                        "search_queries": [
                            "Guardrails Guard.use custom validator metadata"
                        ],
                    },
                )
                if result.isError:
                    raise RuntimeError(f"web_search failed: {result.content}")
                response = result.structuredContent
                if response is None:
                    # MCP also permits the tool's JSON result as text content.
                    text = next(
                        item.text for item in result.content if item.type == "text"
                    )
                    response = json.loads(text)
                # Missing result/URL fields raise rather than becoming an empty retrieval.
                source_urls_from(response)
                return response


if RUN_LIVE_SEARCH:
    live_response = await search_public_docs()
    live_urls = source_urls_from(live_response)
    if not live_urls:
        raise ValueError("Search returned no results; no citation can pass.")
    print(f"Retrieved {len(live_urls)} public source URLs.")
else:
    print("Live search skipped. Set RUN_LIVE_SEARCH = True to opt in.")

In [ ]:
if RUN_LIVE_SEARCH:
    # Constructed candidates, not generated answers: exercise the actual live URL set.
    live_url = live_urls[0]
    live_result = guard.parse(
        answer_json([live_url]), metadata={"source_urls": live_urls}, num_reasks=0
    )
    assert live_result.validation_passed
    assert live_result.validated_output["citations"] == [live_url]

    # A URL absent from this response must fail, even on the returned URL's host.
    live_invented_url = live_url + "/not-returned-by-search"
    assert live_invented_url not in live_urls
    assert not guard.parse(
        answer_json([live_invented_url]),
        metadata={"source_urls": live_urls},
        num_reasks=0,
    ).validation_passed
    print(f"Accepted returned URL: {live_url}")
    print(f"Rejected invented URL: {live_invented_url}")